In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from itertools import product
from functools import partial
import importlib

import src.data
import src.kernels
import src.klr
import src.svm
import src.metrics
import src.preprocessing
import src.utils
from src.data import encode_labels, load_data, train_val_split

from src.kernels import (
    linear_kernel_matrix,
    estimate_gamma,
    gaussian_kernel_matrix,
)
from src.klr import KernelLogisticRegression
from src.svm import KernelSVM
from src.metrics import accuracy
from src.preprocessing import StandardiseData, PCA
from src.utils import show_vector_image

In [2]:
DATA_DIR = "data/"

X_tr, X_te, y_tr_raw = load_data(DATA_DIR)

y_tr, inv_map = encode_labels(y_tr_raw)

scaler = StandardiseData().fit(X_tr.astype(np.float32))
X_tr_s = scaler.transform(X_tr.astype(np.float32))
X_te_s = scaler.transform(X_te.astype(np.float32))

pca = PCA(n_components=512, whiten=False).fit(X_tr.astype(np.float32))
X_tr_p = pca.transform(X_tr_s.astype(np.float32))
X_te_p = pca.transform(X_te_s.astype(np.float32))

n_classes = len(np.unique(y_tr))

X_train, X_val, y_train, y_val = train_val_split(X_tr_p, y_tr, test_size=0.3, seed=20)

### Linear kernel

Quick grid search over learning rate and regularisation.

Kernel Logistic Regression

In [3]:
lrs = [0.3, 0.2, 0.1]
lams = [1e-5, 1e-4, 1e-3]

best_lr, best_lam = lrs[0], lams[0]
best_acc = 0.0

for lr, lam in product(lrs, lams):
    linear_model = KernelLogisticRegression(
        n_classes=n_classes,
        kernel_fn=linear_kernel_matrix,
        lr=lr,
        epochs=200,
        lam=lam,
        patience=5,
    ).fit(X_train, y_train, X_val=X_val, y_val=y_val)

    pred_val, _ = linear_model.predict(X_val)
    acc = accuracy(y_val, pred_val)

    print(f"lr={lr:.1e}, lambda={lam:.1e}, val_acc={acc:.4f}")

    if acc > best_acc:
        best_acc = acc
        best_lr, best_lam = lr, lam

print("best:", {"lr": best_lr, "lambda": best_lam, "val_acc": best_acc})

lr=3.0e-01, lambda=1.0e-05, val_acc=0.1613
lr=3.0e-01, lambda=1.0e-04, val_acc=0.1607
lr=3.0e-01, lambda=1.0e-03, val_acc=0.1667
lr=2.0e-01, lambda=1.0e-05, val_acc=0.1620
lr=2.0e-01, lambda=1.0e-04, val_acc=0.1613
lr=2.0e-01, lambda=1.0e-03, val_acc=0.1647
lr=1.0e-01, lambda=1.0e-05, val_acc=0.1620
lr=1.0e-01, lambda=1.0e-04, val_acc=0.1627
lr=1.0e-01, lambda=1.0e-03, val_acc=0.1647
best: {'lr': 0.3, 'lambda': 0.001, 'val_acc': 0.16666666666666666}


Kernel SVM

In [5]:
lrs = [0.3, 0.2, 0.1]
lams = [1e-5, 1e-4, 1e-3]

best_lr, best_lam = lrs[0], lams[0]
best_acc = 0.0

for lr, lam in product(lrs, lams):
    linear_model = KernelSVM(
        n_classes=n_classes,
        kernel_fn=linear_kernel_matrix,
        lr=lr,
        epochs=200,
        lam=lam,
        patience=5,
    ).fit(X_train, y_train, X_val=X_val, y_val=y_val)

    pred_val, _ = linear_model.predict(X_val)
    acc = accuracy(y_val, pred_val)

    print(f"lr={lr:.1e}, lambda={lam:.1e}, val_acc={acc:.4f}")

    if acc > best_acc:
        best_acc = acc
        best_lr, best_lam = lr, lam

print("best:", {"lr": best_lr, "lambda": best_lam, "val_acc": best_acc})

lr=3.0e-01, lambda=1.0e-05, val_acc=0.1700
lr=3.0e-01, lambda=1.0e-04, val_acc=0.1700
lr=3.0e-01, lambda=1.0e-03, val_acc=0.1647
lr=2.0e-01, lambda=1.0e-05, val_acc=0.1713
lr=2.0e-01, lambda=1.0e-04, val_acc=0.1713
lr=2.0e-01, lambda=1.0e-03, val_acc=0.1647
lr=1.0e-01, lambda=1.0e-05, val_acc=0.1660
lr=1.0e-01, lambda=1.0e-04, val_acc=0.1660
lr=1.0e-01, lambda=1.0e-03, val_acc=0.1640
best: {'lr': 0.2, 'lambda': 1e-05, 'val_acc': 0.17133333333333334}


### Train and write submission (linear)

In [279]:
best_linear_model = KernelLogisticRegression(
    n_classes=n_classes,
    kernel_fn=linear_kernel_matrix,
    lr=best_lr,
    epochs=500,
    lam=best_lam,
).fit(X_tr_p, y_tr)

yte_int, _ = best_linear_model.predict(X_te_p)
yte = np.array([inv_map[i] for i in yte_int])

sub = pd.DataFrame({"Prediction": yte})
sub.index += 1
sub.to_csv("results/submission.csv", index_label="Id")

### Gaussian (RBF) kernel

Kernel Logistic Regression

In [ ]:
X_train, X_val, y_train, y_val = train_val_split(X_tr_s, y_tr, test_size=0.2, seed=20)

lrs = [5e-1, 1e-1]
lams = [5e-4, 1e-4, 5e-5]

gamma0 = estimate_gamma(X_train)
gammas = [gamma0 / 2, gamma0, gamma0 * 2]

best_lr, best_lam, best_gamma = lrs[0], lams[0], gammas[0]
best_acc = 0.0

for lr, lam, gamma in product(lrs, lams, gammas):
    kernel_fn = partial(gaussian_kernel_matrix, gamma=gamma)
    model = KernelLogisticRegression(
        n_classes=n_classes,
        kernel_fn=kernel_fn,
        lr=lr,
        epochs=200,
        lam=lam,
        patience=5,
    ).fit(X_train, y_train, X_val=X_val, y_val=y_val)

    pred_val, _ = model.predict(X_val)
    acc = accuracy(y_val, pred_val)

    print(f"lr={lr:.1e}, lambda={lam:.1e}, gamma={gamma:.3e}, val_acc={acc:.4f}")

    if acc > best_acc:
        best_acc = acc
        best_lr, best_lam, best_gamma = lr, lam, gamma

print("best:", {"lr": best_lr, "lambda": best_lam, "gamma": best_gamma, "val_acc": best_acc})

lr=5.0e-01, lambda=5.0e-04, gamma=4.208e-05, val_acc=0.1820
lr=5.0e-01, lambda=1.0e-04, gamma=4.208e-05, val_acc=0.1780
lr=5.0e-01, lambda=5.0e-05, gamma=4.208e-05, val_acc=0.1770
lr=1.0e-01, lambda=5.0e-04, gamma=4.208e-05, val_acc=0.1660
lr=1.0e-01, lambda=1.0e-04, gamma=4.208e-05, val_acc=0.1640
lr=1.0e-01, lambda=5.0e-05, gamma=4.208e-05, val_acc=0.1640
lr=5.0e-01, lambda=5.0e-04, gamma=8.416e-05, val_acc=0.1930
lr=5.0e-01, lambda=1.0e-04, gamma=8.416e-05, val_acc=0.1880
lr=5.0e-01, lambda=5.0e-05, gamma=8.416e-05, val_acc=0.1880
lr=1.0e-01, lambda=5.0e-04, gamma=8.416e-05, val_acc=0.1800
lr=1.0e-01, lambda=1.0e-04, gamma=8.416e-05, val_acc=0.1840
lr=1.0e-01, lambda=5.0e-05, gamma=8.416e-05, val_acc=0.1840
lr=5.0e-01, lambda=5.0e-04, gamma=1.683e-04, val_acc=0.1810
lr=5.0e-01, lambda=1.0e-04, gamma=1.683e-04, val_acc=0.1800
lr=5.0e-01, lambda=5.0e-05, gamma=1.683e-04, val_acc=0.1800
lr=1.0e-01, lambda=5.0e-04, gamma=1.683e-04, val_acc=0.1790
lr=1.0e-01, lambda=1.0e-04, gamma=1.683e

Kernel SVM

In [ ]:
X_train, X_val, y_train, y_val = train_val_split(X_tr_s, y_tr, test_size=0.2, seed=20)

lrs = [5e-1, 1e-1]
lams = [5e-4, 1e-4, 5e-5]

gamma0 = estimate_gamma(X_train)
gammas = [gamma0 / 5, gamma0 / 2, gamma0]

best_lr, best_lam, best_gamma = lrs[0], lams[0], gammas[0]
best_acc = 0.0

for lr, lam, gamma in product(lrs, lams, gammas):
    kernel_fn = partial(gaussian_kernel_matrix, gamma=gamma)
    model = KernelSVM(
        n_classes=n_classes,
        kernel_fn=kernel_fn,
        lr=lr,
        epochs=200,
        lam=lam,
        patience=5,
    ).fit(X_train, y_train, X_val=X_val, y_val=y_val)

    pred_val, _ = model.predict(X_val)
    acc = accuracy(y_val, pred_val)

    print(f"lr={lr:.1e}, lambda={lam:.1e}, gamma={gamma:.3e}, val_acc={acc:.4f}")

    if acc > best_acc:
        best_acc = acc
        best_lr, best_lam, best_gamma = lr, lam, gamma

print(
    "best:",
    {"lr": best_lr, "lambda": best_lam, "gamma": best_gamma, "val_acc": best_acc},
)

lr=5.0e-01, lambda=5.0e-04, gamma=1.683e-05, val_acc=0.1890
lr=5.0e-01, lambda=1.0e-04, gamma=1.683e-05, val_acc=0.1940
lr=5.0e-01, lambda=5.0e-05, gamma=1.683e-05, val_acc=0.1950
lr=1.0e-01, lambda=5.0e-04, gamma=1.683e-05, val_acc=0.1900
lr=1.0e-01, lambda=1.0e-04, gamma=1.683e-05, val_acc=0.1950
lr=1.0e-01, lambda=5.0e-05, gamma=1.683e-05, val_acc=0.1950
lr=5.0e-01, lambda=5.0e-04, gamma=4.208e-05, val_acc=0.2000
lr=5.0e-01, lambda=1.0e-04, gamma=4.208e-05, val_acc=0.1990
lr=5.0e-01, lambda=5.0e-05, gamma=4.208e-05, val_acc=0.1980
lr=1.0e-01, lambda=5.0e-04, gamma=4.208e-05, val_acc=0.2010
lr=1.0e-01, lambda=1.0e-04, gamma=4.208e-05, val_acc=0.1990
lr=1.0e-01, lambda=5.0e-05, gamma=4.208e-05, val_acc=0.1960
lr=5.0e-01, lambda=5.0e-04, gamma=8.416e-05, val_acc=0.1890
lr=5.0e-01, lambda=1.0e-04, gamma=8.416e-05, val_acc=0.1860
lr=5.0e-01, lambda=5.0e-05, gamma=8.416e-05, val_acc=0.1900
lr=1.0e-01, lambda=5.0e-04, gamma=8.416e-05, val_acc=0.1880
lr=1.0e-01, lambda=1.0e-04, gamma=8.416e

### Train and write submission (Gaussian)

In [280]:
best_gaussian_model = KernelLogisticRegression(
    n_classes=n_classes,
    kernel_fn=partial(gaussian_kernel_matrix, gamma=best_gamma),
    lr=best_lr,
    epochs=500,
    lam=best_lam,
).fit(X_tr_p, y_tr)

yte_int, _ = best_gaussian_model.predict(X_te_p)
yte = np.array([inv_map[i] for i in yte_int])

sub = pd.DataFrame({"Prediction": yte})
sub.index += 1
sub.to_csv("results/resubmission.csv", index_label="Id")